In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import Tensor

print('PyTorch version:', torch.__version__)

PyTorch version: 2.14.0


In [3]:
#dropout的torch实现：手写与torch自带的dropout对比
#手写dropout
def dropout(x: Tensor, #激活值x
            p:float=0.5,
            training:bool=True) -> Tensor:
    if not 0.0<=p<=1.0:
        raise ValueError(f'p must be between 0 and 1, but got {p}')
    if not training or p==0.0:
        return x
    if p==1.0:
        return torch.zeros_like(x)

    keep=1.0-p
    mask=torch.rand_like(x)<keep #torch.rand_like(x)生成一个和激活值 x 形状一样的张量，里面的值是 [0, 1) 之间均匀分布的随机浮点数。< keep若该张量里面元素小于keep保留概率，则变为1。mask就此形成
    return x*mask/keep

#手写与torch自带的dropout对比
x=torch.randn(10,dtype=torch.float32)

actual=dropout(x,p=0.25) #隐藏某个激活值的概率为0.25
expected=F.dropout(x,p=0.25)

print('Custom Dropout:', actual)
print('Custom Dropout:', expected)

Custom Dropout: tensor([-2.6396,  1.5672, -1.0518,  0.3120, -0.1354, -1.1594, -1.6002,  0.0000,
         2.3145, -0.5893])
Custom Dropout: tensor([-2.6396,  1.5672, -1.0518,  0.3120, -0.1354, -1.1594, -0.0000,  0.6766,
         2.3145, -0.5893])


In [4]:
#开启/关闭训练模式&推理模式
dropout=nn.Dropout(p=0.5)
x=torch.ones(10) #激活特征

#训练阶段：关闭部分激活特征
dropout.train()
train_output1=dropout(x)
train_output2=dropout(x)

#推理阶段：开启所有激活特征
dropout.eval()
eval_output1=dropout(x)
eval_output2=dropout(x)

print('Training output1:',train_output1)
print('Training output2:',train_output2)
print('Evaluation output1:',eval_output1)
print('Evaluation output2:',eval_output2)

Training output1: tensor([0., 0., 0., 0., 2., 0., 0., 2., 2., 0.])
Training output2: tensor([2., 0., 0., 0., 2., 2., 2., 0., 2., 0.])
Evaluation output1: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
Evaluation output2: tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])
